# Analyze Saved Benchmark Runs
Load saved result folders and create compact holdout and learning plots.


## Setup
Choose the run directory, then load summary and history files.


In [ ]:
from pathlib import Path
import json
import os
import re
import sys

mpl_config = Path("/tmp/matplotlib")
mpl_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config))

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
repo_root = next(path for path in candidates if (path / "src" / "qvnn_pc").exists())
sys.path.insert(0, str(repo_root / "src"))

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display


In [ ]:
RUN_NAME = "cpcn-h200"
SUMMARY_FILES = ("cpcn_benchmark.csv", "architecture_comparison.csv")
RUN_CONFIG_FILE = "run_config.json"
SHOW_FUNCTIONS = None  # Example: ["ackley", "rastrigin"]
SHOW_MODELS = None     # Example: ["CPCN", "CMLP", "GHR"]
FIG_DPI = 120


In [ ]:
def _iter_run_sections(run_dir: Path) -> list[Path]:
    if any((run_dir / filename).exists() for filename in SUMMARY_FILES):
        return [run_dir]

    section_dirs = [
        path
        for path in sorted(run_dir.iterdir())
        if path.is_dir()
        and (
            any((path / filename).exists() for filename in SUMMARY_FILES)
            or (path / RUN_CONFIG_FILE).exists()
            or (path / "history").exists()
            or (path / "models").exists()
        )
    ]
    return section_dirs or [run_dir]


def _load_summary_frame(section_dir: Path) -> pd.DataFrame | None:
    for filename in SUMMARY_FILES:
        summary_path = section_dir / filename
        if summary_path.exists():
            summary_df = pd.read_csv(summary_path)
            summary_df["source_run"] = section_dir.name
            return summary_df
    return None


def _load_run_summary(run_dir: Path) -> tuple[pd.DataFrame, list[Path]]:
    section_dirs = _iter_run_sections(run_dir)
    frames: list[pd.DataFrame] = []
    skipped_sections: list[Path] = []

    for section_dir in section_dirs:
        section_summary = _load_summary_frame(section_dir)
        if section_summary is None:
            skipped_sections.append(section_dir)
            continue

        frames.append(section_summary)

    if not frames:
        # Return an empty summary DataFrame instead of raising so the notebook
        # can continue and build plots from available history files.
        cols = [
            "test_function",
            "algorithm",
            "repetition",
            "test_mape",
            "test_r2",
            "runtime_seconds",
            "source_run",
        ]
        summary_df = pd.DataFrame(columns=cols)
    else:
        summary_df = pd.concat(frames, ignore_index=True)

    if "algorithm" in summary_df.columns:
        summary_df["algorithm"] = summary_df["algorithm"].astype(str).str.upper()

    return summary_df, skipped_sections


def _extract_history_series(payload: dict, candidate_keys: list[str]) -> list[float]:
    for key in candidate_keys:
        if key in payload:
            return payload[key]
    raise KeyError(f"None of the expected history keys were found: {candidate_keys}")


In [ ]:
cwd = Path.cwd().resolve()
# locate repository root (assumes src/qvnn_pc exists in repo)
candidates = [cwd, *cwd.parents]
repo_root = next(path for path in candidates if (path / "src" / "qvnn_pc").exists())

results_root = repo_root / "results"
available_runs = sorted(path.name for path in results_root.iterdir() if path.is_dir())
run_dir = results_root / RUN_NAME

if not run_dir.exists():
    raise FileNotFoundError(f"Run directory not found: {run_dir}")

# Try to load summary files if present, but don't fail the notebook if missing.
try:
    summary_df, skipped_sections = _load_run_summary(run_dir)
except Exception:
    summary_df = pd.DataFrame()
    skipped_sections = []

print(f"Selected run: {run_dir}")
print(f"Loaded summary rows: {len(summary_df)}")


In [ ]:
if SHOW_FUNCTIONS is not None and not summary_df.empty:
    summary_df = summary_df[summary_df["test_function"].isin(SHOW_FUNCTIONS)].copy()
if SHOW_MODELS is not None and not summary_df.empty:
    summary_df = summary_df[summary_df["algorithm"].isin(SHOW_MODELS)].copy()

if summary_df.empty:
    print("No summary CSVs found (or filters removed all rows); continuing using history files for learning plots.")
    # Initialize empty orders; they will be derived from history files later if available.
    function_order = []
    model_order = []
else:
    preferred_function_order = ["ackley", "griewank", "rastrigin", "schwefel"]
    function_order = [name for name in preferred_function_order if name in set(summary_df["test_function"])]
    function_order += [name for name in sorted(summary_df["test_function"].unique()) if name not in function_order]

    preferred_model_order = ["CPCN", "CMLP", "GHR", "Arena", "Nitta"]
    model_order = [name for name in preferred_model_order if name in set(summary_df["algorithm"])]
    model_order += [name for name in sorted(summary_df["algorithm"].unique()) if name not in model_order]

    summary_df["test_function"] = pd.Categorical(summary_df["test_function"], categories=function_order, ordered=True)
    summary_df["algorithm"] = pd.Categorical(summary_df["algorithm"], categories=model_order, ordered=True)
    summary_df = summary_df.sort_values(["test_function", "algorithm", "repetition"]).reset_index(drop=True)

summary_df


## Summary Tables
Compare holdout metrics and runtime per function/model.


In [ ]:
def _format_mean_std(mean_value: float, std_value: float, digits: int = 4) -> str:
    if pd.isna(std_value):
        std_value = 0.0
    return f"{mean_value:.{digits}f} +/- {std_value:.{digits}f}"

if summary_df.empty:
    display(Markdown("**No summary CSV available — skipping holdout comparison tables.**"))
else:
    summary_stats = (
        summary_df.groupby(["test_function", "algorithm"], observed=True)
        .agg(
            runs=("repetition", "count"),
            mape_mean=("test_mape", "mean"),
            mape_std=("test_mape", "std"),
            r2_mean=("test_r2", "mean"),
            r2_std=("test_r2", "std"),
            runtime_mean=("runtime_seconds", "mean"),
            runtime_std=("runtime_seconds", "std"),
        )
        .reset_index()
    )

    summary_stats["MAPE"] = summary_stats.apply(lambda row: _format_mean_std(row.mape_mean, row.mape_std), axis=1)
    summary_stats["R^2"] = summary_stats.apply(lambda row: _format_mean_std(row.r2_mean, row.r2_std), axis=1)
    summary_stats["Runtime (s)"] = summary_stats.apply(lambda row: _format_mean_std(row.runtime_mean, row.runtime_std, digits=2), axis=1)

    tidy_table = summary_stats[["test_function", "algorithm", "runs", "MAPE", "R^2", "Runtime (s)"]]
    display(tidy_table)

    pivot_table = (
        summary_stats.set_index(["test_function", "algorithm"])[["MAPE", "R^2", "Runtime (s)"]]
        .unstack("algorithm")
    )
    display(pivot_table)


## Plot Helpers
Shared styling for the figures.


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = FIG_DPI

palette = plt.get_cmap("tab10")


def _present_models(df: pd.DataFrame) -> list[str]:
    return [model for model in model_order if model in set(df["algorithm"])]


def _boxplot_with_points(ax, df: pd.DataFrame, metric_col: str, ylabel: str, title: str):
    present_models = _present_models(df)
    values = [df.loc[df["algorithm"] == model, metric_col].to_numpy() for model in present_models]
    positions = np.arange(1, len(present_models) + 1)

    # compute colors for present models
    model_colors = {model: palette(i) for i, model in enumerate(present_models)}

    box = ax.boxplot(values, positions=positions, widths=0.6, patch_artist=True, tick_labels=present_models)
    for patch, model in zip(box["boxes"], present_models):
        patch.set_facecolor(model_colors[model])
        patch.set_alpha(0.45)
    for median in box["medians"]:
        median.set_color("black")
        median.set_linewidth(1.6)

    for xpos, model, series in zip(positions, present_models, values):
        if len(series) == 0:
            continue
        offsets = np.linspace(-0.07, 0.07, num=len(series)) if len(series) > 1 else np.array([0.0])
        ax.scatter(
            np.full(len(series), xpos) + offsets,
            series,
            color=model_colors[model],
            edgecolor="black",
            linewidth=0.5,
            s=34,
            zorder=3,
        )

    ax.set_title(title)
    ax.set_xlabel("Model")
    ax.set_ylabel(ylabel)


def plot_function_comparison(function_name: str, df: pd.DataFrame):
    function_df = df[df["test_function"] == function_name].copy()
    present_models = _present_models(function_df)
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), squeeze=False)
    axes = axes.ravel()

    _boxplot_with_points(
        axes[0],
        function_df,
        metric_col="test_mape",
        ylabel="MAPE",
        title=f"{function_name.capitalize()} MAPE Results (lower is better)",
    )
    _boxplot_with_points(
        axes[1],
        function_df,
        metric_col="runtime_seconds",
        ylabel="Runtime (seconds)",
        title=f"{function_name.capitalize()} Runtime Results (lower is better)",
    )
    _boxplot_with_points(
        axes[2],
        function_df,
        metric_col="test_r2",
        ylabel="R^2",
        title=f"{function_name.capitalize()} R^2 Results (higher is better)",
    )

    # compute colors for present models
    model_colors = {model: palette(i) for i, model in enumerate(present_models)}
    for model in present_models:
        model_df = function_df[function_df["algorithm"] == model]
        axes[3].scatter(
            model_df["test_mape"],
            model_df["test_r2"],
            color=model_colors[model],
            s=60,
            alpha=0.9,
            edgecolor="black",
            linewidth=0.5,
            label=model,
        )
    axes[3].set_title(f"{function_name.capitalize()} MAPE vs R^2")
    axes[3].set_xlabel("MAPE")
    axes[3].set_ylabel("R^2")
    axes[3].legend(title="Model")

    fig.suptitle(f"Holdout Comparison for {function_name.capitalize()}", fontsize=15, y=1.02)
    fig.tight_layout()
    return fig


## Holdout Comparison
Compare MAPE, R2, runtime, and the MAPE/R2 trade-off.


In [ ]:
if summary_df.empty:
    display(Markdown("**No summary CSV available — skipping holdout comparison figures.**"))
else:
    for function_name in function_order:
        display(Markdown(f"### {function_name.capitalize()}"))
        plot_function_comparison(function_name, summary_df)
        plt.show()


## Learning Data
Collect per-epoch train and validation MAE from history files.


In [ ]:
history_pattern = re.compile(r"(?P<function>.+)_(?P<algorithm>[^_]+)_rep(?P<rep>\d+)\.json")

history_rows = []
for section_dir in _iter_run_sections(run_dir):
    history_dir = section_dir / "history"
    if not history_dir.exists():
        continue

    for history_path in sorted(history_dir.glob("*.json")):
        match = history_pattern.fullmatch(history_path.name)
        if match is None:
            continue
        function_name = match.group("function")
        algorithm_name = match.group("algorithm").upper()
        repetition = int(match.group("rep"))
        payload = json.loads(history_path.read_text())
        train_values = _extract_history_series(payload, ["quaternion_mae", "train_quaternion_mae", "train_mae"])
        validation_values = _extract_history_series(payload, [
            "validation_quaternion_mae",
            "val_quaternion_mae",
            "validation_mae",
            "val_mae",
        ])
        curve_length = min(len(train_values), len(validation_values))

        for epoch_idx in range(curve_length):
            history_rows.append(
                {
                    "test_function": function_name,
                    "algorithm": algorithm_name,
                    "repetition": repetition,
                    "epoch": epoch_idx + 1,
                    "train_mae": train_values[epoch_idx],
                    "validation_mae": validation_values[epoch_idx],
                }
            )

history_df = pd.DataFrame(history_rows)

history_df = pd.DataFrame(history_rows)

if not history_df.empty:
    # If we don't have a summary, derive function/model ordering from history files.
    if not function_order:
        preferred_function_order = ["ackley", "griewank", "rastrigin", "schwefel"]
        function_order = [name for name in preferred_function_order if name in set(history_df["test_function"])]
        function_order += [name for name in sorted(history_df["test_function"].unique()) if name not in function_order]
    if not model_order:
        preferred_model_order = ["CPCN", "CMLP", "GHR", "Arena", "Nitta"]
        model_order = [name for name in preferred_model_order if name in set(history_df["algorithm"])]
        model_order += [name for name in sorted(history_df["algorithm"].unique()) if name not in model_order]

    history_df["test_function"] = pd.Categorical(history_df["test_function"], categories=function_order, ordered=True)
    history_df["algorithm"] = pd.Categorical(history_df["algorithm"], categories=model_order, ordered=True)
    history_df = history_df.sort_values(["test_function", "algorithm", "repetition", "epoch"]).reset_index(drop=True)

history_df.head()


In [ ]:
def plot_learning_behaviour_for_function(function_name: str, history_df: pd.DataFrame):
    function_df = history_df[history_df["test_function"] == function_name].copy()
    present_models = _present_models(function_df)

    fig, axes = plt.subplots(1, len(present_models), figsize=(6.2 * len(present_models), 4.6), squeeze=False)
    axes = axes.ravel()

    for ax, model_name in zip(axes, present_models):
        subset = function_df[function_df["algorithm"] == model_name]
        sns.lineplot(
            data=subset, 
            x="epoch", 
            y="train_mae", 
            estimator="mean", 
            errorbar="se", 
            ax=ax, 
            color="tab:blue", 
            label="Train mean"
        )
        sns.lineplot(
            data=subset, 
            x="epoch", 
            y="validation_mae", 
            estimator="mean", 
            errorbar="se", 
            ax=ax, 
            color="tab:orange", 
            linestyle="--", 
            label="Validation mean"
        )
        ax.set_title(f"{function_name.capitalize()} - {model_name}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MAE")

    legend_handles = [
        Line2D([0], [0], color="tab:blue", linewidth=2.2, label="Train mean"),
        Line2D([0], [0], color="tab:orange", linewidth=2.2, linestyle="--", label="Validation mean"),
    ]
    fig.legend(handles=legend_handles, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.04))
    fig.suptitle(f"Learning Behaviour for {function_name.capitalize()}", fontsize=15, y=1.08)
    fig.tight_layout()
    return fig


## Learning Curves
Plot mean and standard deviation across repetitions.


In [ ]:
for function_name in function_order:
    plot_learning_behaviour_for_function(function_name, history_df)
    plt.show()
